# RateTransitionComponent

This notebook showcases the **RateTransitionComponent** class for multi-rate control loop modeling. It models sample-rate changes between different parts of a feedback loop (e.g., ADC path at 40 MHz → loop path at 20 MHz, or loop at 20 MHz → VCO path at 40 MHz).

## Expected Behaviour

| Aspect | Downsampling (sps_in > sps_out) | Upsampling (sps_in < sps_out) |
|--------|--------------------------------|-------------------------------|
| **Ratio** | `sps_in / sps_out` must be integer | `sps_out / sps_in` must be integer |
| **DC gain** | Unity (1.0) | Unity (1.0) |
| **Delay** | Causal delay of `(M − 1)` samples at input rate | ZOH: ~0.5 sample at output rate |
| **Component sps** | `sps_in` (input rate) | `sps_out` (output rate) |

**Typical use:** In optical PLLs, the ADC/sensor path runs at 40 MHz while the loop filter runs at 20 MHz (downsampling). The VCO (PA + LUT) may run at 40 MHz with the control signal at 20 MHz (upsampling).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from loopkit.components import RateTransitionComponent

## Create RateTransitionComponent

Example: 40 MHz → 20 MHz downsampling (M=2), as used in the optical PLL.

In [ ]:
sps_in = 40e6   # ADC/sensor path (Hz)
sps_out = 20e6  # Loop path (Hz)

rt = RateTransitionComponent(
    "RateTransition",
    sps_in=sps_in,
    sps_out=sps_out,
    include_delay=True,
)

print(f"Downsample factor M = {rt.M}")
print(f"Delay = {rt.M - 1} sample(s) at input rate ({sps_in/1e6:.0f} MHz)")
print(f"Component sps (for TF eval) = {rt.sps:.0f} Hz")

## Bode Plot

The transfer function is evaluated at the **input** sample rate. For 40→20 MHz with `include_delay=True`, the component adds a 1-sample delay at 40 MHz.

In [ ]:
# Frequency range: up to Nyquist at input rate
freq = np.logspace(2, np.log10(sps_in / 2), 500)

ax_mag, ax_phase = rt.bode_plot(freq, dB=True, figsize=(6, 5), title="RateTransition 40→20 MHz")
plt.show()

## With vs Without Delay

Compare `include_delay=True` (default) with `include_delay=False`. The delay adds phase lag; without it, the component is pure unity gain.

In [ ]:
rt_no_delay = RateTransitionComponent(
    "RateTransition (no delay)",
    sps_in=sps_in,
    sps_out=sps_out,
    include_delay=False,
)

fig, axes = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
ax_mag, ax_phase = axes

rt.bode_plot(freq, dB=True, axes=axes, label="include_delay=True")
rt_no_delay.bode_plot(freq, dB=True, axes=axes, label="include_delay=False")

ax_mag.set_title("RateTransition: effect of include_delay")
plt.tight_layout()
plt.show()

## Validation

- **DC magnitude:** Should be 0 dB (unity) for both configurations.
- **Phase with delay:** At 40 MHz, 1 sample delay = 360° at 40 MHz; phase rolls off linearly with frequency.
- **Phase without delay:** Should remain 0°.
- **At Nyquist** (f = sps_in/2 = 20 MHz): `include_delay=True` → 0 dB, −180°; `include_delay=False` → 0 dB, 0°.

In [ ]:
mag_rt, phase_rt = rt.bode(freq, dB=True)
mag_nd, phase_nd = rt_no_delay.bode(freq, dB=True)

print("DC (lowest freq) magnitude:")
print(f"  include_delay=True:  {mag_rt[0]:.4f} dB")
print(f"  include_delay=False: {mag_nd[0]:.4f} dB")
print()
print("DC phase:")
print(f"  include_delay=True:  {phase_rt[0]:.2f} deg")
print(f"  include_delay=False: {phase_nd[0]:.2f} deg")

# Nyquist verification (f = sps_in/2)
f_nyq = sps_in / 2
idx_nyq = np.argmin(np.abs(freq - f_nyq))
print()
print(f"At Nyquist (f = {f_nyq/1e6:.0f} MHz):")
print(f"  include_delay=True:  {mag_rt[idx_nyq]:.4f} dB, {phase_rt[idx_nyq]:.2f} deg (expected: 0 dB, -180 deg)")
print(f"  include_delay=False: {mag_nd[idx_nyq]:.4f} dB, {phase_nd[idx_nyq]:.2f} deg (expected: 0 dB, 0 deg)")

## Upsampling: 20 MHz → 40 MHz

When `sps_in < sps_out`, the component performs ZOH (zero-order hold) upsampling. Each input sample is held for M output samples.

In [ ]:
rt_up = RateTransitionComponent(
    "Upsample",
    sps_in=20e6,
    sps_out=40e6,
)

print(f"Upsample factor M = {rt_up.M}")
print(f"Component sps (for TF eval) = {rt_up.sps:.0f} Hz (output rate)")
mag, _ = rt_up.bode([1.0], dB=True)
print(f"DC gain = {10**(mag[0]/20):.4f} (0 dB)")

In [ ]:
freq_up = np.logspace(2, np.log10(40e6 / 2), 500)
rt_up.bode_plot(freq_up, dB=True, figsize=(6, 5), title="RateTransition 20→40 MHz (ZOH upsample)")
plt.show()